In [1]:
pip install pandas scikit-learn openpyxl


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
import pandas as pd
from sklearn.model_selection import StratifiedKFold

# =========================================================
# 1) Excel file path (ဥပမာ - ကိုယ်တိုင်ပြင်)
# =========================================================
file_path = "/Users/shinekhantaung/Desktop/GPT-NER/D_LLMs.xlsx"

# =========================================================
# 2) Output folder path (ဥပမာ - ကိုယ်တိုင်ပြင်)
# =========================================================
output_dir = "/Users/shinekhantaung/Desktop/GPT-NER/5fold_output"

# folder မရှိရင် create လုပ်မယ်
os.makedirs(output_dir, exist_ok=True)

# =========================================================
# 3) Read Excel
# =========================================================
df = pd.read_excel(file_path)

sentence_col = "sentence"
name_col = "gold_names"

# =========================================================
# 4) Clean label column
# =========================================================
df[name_col] = df[name_col].fillna("").astype(str).str.strip()
df["has_name"] = (df[name_col] != "").astype(int)

print("========== Overall Dataset Info ==========")
print(f"Total rows           : {len(df)}")
print(f"Rows with name       : {(df['has_name'] == 1).sum()}")
print(f"Rows without name    : {(df['has_name'] == 0).sum()}")
print()

# =========================================================
# 5) Stratified 5-fold
# =========================================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

df["fold"] = -1
X = df.index
y = df["has_name"]

for fold_id, (_, valid_idx) in enumerate(skf.split(X, y), start=1):
    df.loc[valid_idx, "fold"] = fold_id

# =========================================================
# 6) Fold stats
# =========================================================
print("========== Fold-wise Info ==========")
for fold_id in range(1, 6):
    fold_df = df[df["fold"] == fold_id]
    print(f"Fold {fold_id}")
    print(f"  Total rows        : {len(fold_df)}")
    print(f"  Rows with name    : {(fold_df['has_name'] == 1).sum()}")
    print(f"  Rows without name : {(fold_df['has_name'] == 0).sum()}")
    print()

# =========================================================
# 7) Save each fold
# =========================================================
for fold_id in range(1, 6):
    fold_df = df[df["fold"] == fold_id].copy()
    out_path = os.path.join(output_dir, f"fold_{fold_id}.xlsx")
    fold_df.to_excel(out_path, index=False)

print(f"Saved 5 fold files to: {output_dir}")

========== Overall Dataset Info ==========
Total rows           : 929
Rows with name       : 904
Rows without name    : 25

========== Fold-wise Info ==========
Fold 1
  Total rows        : 186
  Rows with name    : 181
  Rows without name : 5

Fold 2
  Total rows        : 186
  Rows with name    : 181
  Rows without name : 5

Fold 3
  Total rows        : 186
  Rows with name    : 181
  Rows without name : 5

Fold 4
  Total rows        : 186
  Rows with name    : 181
  Rows without name : 5

Fold 5
  Total rows        : 185
  Rows with name    : 180
  Rows without name : 5

Saved 5 fold files to: /Users/shinekhantaung/Desktop/GPT-NER/5fold_output


In [6]:
import os
import pandas as pd
from sklearn.model_selection import StratifiedKFold

# =========================================================
# 1) File path
# =========================================================
file_path = "/Users/shinekhantaung/Desktop/GPT-NER/D_LLMs.xlsx"

# =========================================================
# 2) Output folder path 
# =========================================================
output_dir = "/Users/shinekhantaung/Desktop/GPT-NER/5fold_train_test"

# output folder မရှိရင် create
os.makedirs(output_dir, exist_ok=True)

# =========================================================
# 2) Read Excel
# =========================================================
df = pd.read_excel(file_path)

# column names (dataset နဲ့ကိုက်အောင်ပြင်)
sentence_col = "sentence"
name_col = "gold_names"

# =========================================================
# 3) Clean name column
#    blank / NaN / spaces only => no-name
# =========================================================
df[name_col] = df[name_col].fillna("").astype(str).str.strip()
df["has_name"] = (df[name_col] != "").astype(int)

print("========== Overall Dataset Info ==========")
print(f"Total rows           : {len(df)}")
print(f"Rows with name       : {(df['has_name'] == 1).sum()}")
print(f"Rows without name    : {(df['has_name'] == 0).sum()}")
print()

# =========================================================
# 4) Make 5 folds
#    no-name 25 rows => each fold gets 5 no-name rows
# =========================================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

df["fold"] = -1
X = df.index
y = df["has_name"]

for fold_id, (_, test_idx) in enumerate(skf.split(X, y), start=1):
    df.loc[test_idx, "fold"] = fold_id

# =========================================================
# 5) Print fold-wise stats
# =========================================================
print("========== Fold-wise Info ==========")
for fold_id in range(1, 6):
    fold_df = df[df["fold"] == fold_id]
    print(f"Fold {fold_id}")
    print(f"  Total rows        : {len(fold_df)}")
    print(f"  Rows with name    : {(fold_df['has_name'] == 1).sum()}")
    print(f"  Rows without name : {(fold_df['has_name'] == 0).sum()}")
    print()

# =========================================================
# 6) For each fold:
#    test = current fold
#    train = all remaining folds
# =========================================================
print("========== Train / Test Split Info ==========")

for fold_id in range(1, 6):
    test_df = df[df["fold"] == fold_id].copy()
    train_df = df[df["fold"] != fold_id].copy()

    # stats
    train_total = len(train_df)
    train_no_name = (train_df["has_name"] == 0).sum()
    train_has_name = (train_df["has_name"] == 1).sum()

    test_total = len(test_df)
    test_no_name = (test_df["has_name"] == 0).sum()
    test_has_name = (test_df["has_name"] == 1).sum()

    print(f"Fold {fold_id}")
    print(f"  Train rows              : {train_total}")
    print(f"  Train rows with name    : {train_has_name}")
    print(f"  Train rows without name : {train_no_name}")
    print(f"  Test rows               : {test_total}")
    print(f"  Test rows with name     : {test_has_name}")
    print(f"  Test rows without name  : {test_no_name}")
    print()

    # save files
    train_out = os.path.join(output_dir, f"train_fold_{fold_id}.xlsx")
    test_out = os.path.join(output_dir, f"test_fold_{fold_id}.xlsx")

    train_df.to_excel(train_out, index=False)
    test_df.to_excel(test_out, index=False)

print(f"All train/test files saved in: {output_dir}")

========== Overall Dataset Info ==========
Total rows           : 929
Rows with name       : 904
Rows without name    : 25

========== Fold-wise Info ==========
Fold 1
  Total rows        : 186
  Rows with name    : 181
  Rows without name : 5

Fold 2
  Total rows        : 186
  Rows with name    : 181
  Rows without name : 5

Fold 3
  Total rows        : 186
  Rows with name    : 181
  Rows without name : 5

Fold 4
  Total rows        : 186
  Rows with name    : 181
  Rows without name : 5

Fold 5
  Total rows        : 185
  Rows with name    : 180
  Rows without name : 5

========== Train / Test Split Info ==========
Fold 1
  Train rows              : 743
  Train rows with name    : 723
  Train rows without name : 20
  Test rows               : 186
  Test rows with name     : 181
  Test rows without name  : 5

Fold 2
  Train rows              : 743
  Train rows with name    : 723
  Train rows without name : 20
  Test rows               : 186
  Test rows with name     : 181
  Test row